In [7]:
# =====================================================
# 12_final_feature_engineering.ipynb
# =====================================================
# Combines old and new feature logic — including ADX, CCI, Stochastic, Williams %R —
# while importing NIFTY contextual features from 07_advanced_feature_engineering.

import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

# =====================================================
# Project Paths
# =====================================================
project_root = Path("C:/JupyterProjects/Stock_ML_Project").resolve()
processed_dir = project_root / "Data" / "Processed"
enhanced_dir = processed_dir / "enhanced"
enhanced_dir.mkdir(parents=True, exist_ok=True)

print("Using processed_dir:", processed_dir)
print("Enhanced (final) will be saved in:", enhanced_dir)

# Add notebooks path to import old feature logic
sys.path.append(str(project_root / "Notebooks"))

# Import the previously validated index merger
from feature_utils import add_index_features

# =====================================================
# Config Files
# =====================================================
files = {
    "RELIANCE": processed_dir / "reliance_model_ready.csv",
    "TCS": processed_dir / "tcs_model_ready.csv",
    "HDFCBANK": processed_dir / "hdfcbank_model_ready.csv",
}

# =====================================================
# Helper Functions
# =====================================================
def compute_RSI(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))


def add_basic_features(df, prefix, col):
    df[f"{prefix}_Return_3"] = df[col].pct_change(3)
    df[f"{prefix}_Return_5"] = df[col].pct_change(5)
    df[f"{prefix}_Return_10"] = df[col].pct_change(10)
    df[f"{prefix}_Volatility_7"] = df[col].pct_change().rolling(7).std()
    df[f"{prefix}_Volatility_14"] = df[col].pct_change().rolling(14).std()
    df[f"{prefix}_Momentum_5"] = df[col] - df[col].shift(5)
    df[f"{prefix}_Momentum_10"] = df[col] - df[col].shift(10)
    return df


def add_bollinger_bands(df, prefix, col="Close", window=20):
    mean = df[col].rolling(window).mean()
    std = df[col].rolling(window).std()
    df[f"{prefix}_BB_upper"] = mean + (2 * std)
    df[f"{prefix}_BB_lower"] = mean - (2 * std)
    df[f"{prefix}_BB_width"] = df[f"{prefix}_BB_upper"] - df[f"{prefix}_BB_lower"]
    return df


def add_advanced_indicators(df, prefix, col="Close"):
    """Adds MACD, ADX, Stochastic, Williams %R, CCI, etc."""
    # MACD
    df[f"{prefix}_EMA12"] = df[col].ewm(span=12, adjust=False).mean()
    df[f"{prefix}_EMA26"] = df[col].ewm(span=26, adjust=False).mean()
    df[f"{prefix}_MACD"] = df[f"{prefix}_EMA12"] - df[f"{prefix}_EMA26"]
    df[f"{prefix}_Signal"] = df[f"{prefix}_MACD"].ewm(span=9, adjust=False).mean()

    # --- Ensure High/Low exist ---
    if "High" not in df.columns or "Low" not in df.columns:
        print(f"  ⚠️ Synthesizing 'High' and 'Low' for {prefix}...")
        df["High"] = df[col] * (1 + np.random.uniform(0.002, 0.005, size=len(df)))
        df["Low"] = df[col] * (1 - np.random.uniform(0.002, 0.005, size=len(df)))

    # --- ADX ---
    df["H-L"] = df["High"] - df["Low"]
    df["H-PC"] = abs(df["High"] - df["Close"].shift(1))
    df["L-PC"] = abs(df["Low"] - df["Close"].shift(1))
    df["TR"] = df[["H-L", "H-PC", "L-PC"]].max(axis=1)
    df["DM_plus"] = np.where((df["High"] - df["High"].shift(1)) > (df["Low"].shift(1) - df["Low"]),
                             df["High"] - df["High"].shift(1), 0)
    df["DM_minus"] = np.where((df["Low"].shift(1) - df["Low"]) > (df["High"] - df["High"].shift(1)),
                              df["Low"].shift(1) - df["Low"], 0)
    TRn = df["TR"].rolling(14).sum()
    DMPn = df["DM_plus"].rolling(14).sum()
    DMNn = df["DM_minus"].rolling(14).sum()
    DIp = 100 * (DMPn / TRn)
    DIn = 100 * (DMNn / TRn)
    DX = (abs(DIp - DIn) / abs(DIp + DIn)) * 100
    df[f"{prefix}_ADX"] = DX.rolling(14).mean()

    # --- Stochastic Oscillator ---
    df[f"{prefix}_Stochastic"] = 100 * ((df["Close"] - df["Low"].rolling(14).min()) /
                                        (df["High"].rolling(14).max() - df["Low"].rolling(14).min()))

    # --- Williams %R ---
    df[f"{prefix}_WilliamsR"] = -100 * ((df["High"].rolling(14).max() - df["Close"]) /
                                        (df["High"].rolling(14).max() - df["Low"].rolling(14).min()))

    # --- Commodity Channel Index (CCI) ---
    tp = (df["High"] + df["Low"] + df["Close"]) / 3
    ma = tp.rolling(20).mean()
    md = (tp - ma).abs().rolling(20).mean()
    df[f"{prefix}_CCI"] = (tp - ma) / (0.015 * md)

    return df


def add_lag_features(df, prefix, col):
    for lag in [1, 2, 3, 5, 10]:
        df[f"{prefix}_Lag_{lag}"] = df[col].shift(lag)
    return df


def add_rolling_stats(df, prefix, col):
    for w in [7, 21, 50]:
        df[f"{prefix}_MA{w}"] = df[col].rolling(w).mean()
        df[f"{prefix}_STD{w}"] = df[col].rolling(w).std()
    df[f"{prefix}_RSI"] = compute_RSI(df[col], 14)
    return df


# =====================================================
# Main Loop
# =====================================================
for ticker, filepath in files.items():
    print(f"\n=== Processing {ticker} ===")

    if not filepath.exists():
        print(f"  ⚠️ Missing file: {filepath}")
        continue

    df = pd.read_csv(filepath)

    # Handle missing Date
    if "Date" not in df.columns:
        print("  ⚠️ No Date column found. Creating synthetic dates...")
        df["Date"] = pd.date_range(start="2018-01-01", periods=len(df), freq="D")

    # Detect close column dynamically
    close_candidates = [c for c in df.columns if "Close" in c or "close" in c]
    if close_candidates:
        close_col = close_candidates[0]
    else:
        first_num_col = df.select_dtypes(include=[np.number]).columns[0]
        df["Close"] = df[first_num_col]
        close_col = "Close"

    # Add advanced features
    df = add_basic_features(df, ticker, close_col)
    df = add_bollinger_bands(df, ticker, col=close_col)
    df = add_advanced_indicators(df, ticker, col=close_col)
    df = add_lag_features(df, ticker, col=close_col)
    df = add_rolling_stats(df, ticker, col=close_col)

    # Add NIFTY contextual features (imported)
    df = add_index_features(df)

    # Fill missing
    df = df.ffill().bfill()

    # Save final enhanced dataset
    save_path = enhanced_dir / f"{ticker.lower()}_final_model_ready.csv"
    df.to_csv(save_path, index=False)
    print(f"  ✅ Saved enhanced dataset: {save_path}, shape={df.shape}")

print("\n🎯 Final feature engineering completed successfully! Enhanced datasets ready.")


[*********************100%***********************]  1 of 1 completed

Using processed_dir: C:\JupyterProjects\Stock_ML_Project\Data\Processed
Enhanced (final) will be saved in: C:\JupyterProjects\Stock_ML_Project\Data\Processed\enhanced

=== Processing RELIANCE ===
  ⚠️ No Date column found. Creating synthetic dates...
  ⚠️ Synthesizing 'High' and 'Low' for RELIANCE...
  ✅ Saved enhanced dataset: C:\JupyterProjects\Stock_ML_Project\Data\Processed\enhanced\reliance_final_model_ready.csv, shape=(1460, 54)

=== Processing TCS ===
  ⚠️ No Date column found. Creating synthetic dates...
  ⚠️ Synthesizing 'High' and 'Low' for TCS...



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


  ✅ Saved enhanced dataset: C:\JupyterProjects\Stock_ML_Project\Data\Processed\enhanced\tcs_final_model_ready.csv, shape=(1460, 54)

=== Processing HDFCBANK ===
  ⚠️ No Date column found. Creating synthetic dates...
  ⚠️ Synthesizing 'High' and 'Low' for HDFCBANK...
  ✅ Saved enhanced dataset: C:\JupyterProjects\Stock_ML_Project\Data\Processed\enhanced\hdfcbank_final_model_ready.csv, shape=(1460, 54)

🎯 Final feature engineering completed successfully! Enhanced datasets ready.


In [8]:
import pandas as pd

file_path = r"C:\JupyterProjects\Stock_ML_Project\Data\Processed\enhanced\reliance_final_model_ready.csv"
df = pd.read_csv(file_path)

print("✅ Dataset shape:", df.shape)
display(df.sample(5, random_state=42))


✅ Dataset shape: (1460, 54)


,RELIANCE.NS_Return,RELIANCE.NS_MA7,RELIANCE.NS_MA21,RELIANCE.NS_EMA21,RELIANCE.NS_STD21,RELIANCE.NS_RSI,Target_Reg,Target_Cls,Split,Date,...,RELIANCE_MA21,RELIANCE_STD21,RELIANCE_MA50,RELIANCE_STD50,RELIANCE_RSI,NIFTY_Close,NIFTY_Return,NIFTY_MA7,NIFTY_MA21,NIFTY_RSI
892,-0.006227,1173.457153,1189.680664,1179.087322,16.192771,43.553793,1186.001099,1,Train,2020-06-11,...,0.000260,0.011665,0.000527,0.016433,52.819035,9902.000000,-0.021169,10094.100307,9680.314314,67.960232
1105,-0.001211,1274.855416,1212.894345,1228.242480,55.277021,84.154830,1255.613403,0,Train,2021-01-10,...,0.005062,0.013053,0.002628,0.010983,48.091379,14347.250000,0.000000,14236.821429,13946.454753,91.372563
413,-0.000836,1006.950997,1013.945719,1003.779377,31.046642,31.964703,1023.621094,1,Train,2019-02-18,...,0.003139,0.019554,0.000677,0.019426,47.856686,10640.950195,-0.007781,10746.143136,10844.616769,26.835764
522,-0.020326,988.373265,954.567796,955.715126,28.284501,55.989994,956.241150,0,Train,2019-06-07,...,0.003994,0.020703,0.001566,0.019674,46.419796,11870.650391,0.002271,11955.978655,11835.907134,51.902866
1036,0.004947,1058.139614,1039.906924,1051.240265,22.939213,65.118138,1071.237915,1,Train,2020-11-02,...,-0.001497,0.016250,-0.000573,0.014108,51.082103,11669.150391,0.002298,11698.021624,11805.816685,35.387749
